# Milestone 4 — FakeQuantLinear: Linear Replacement + Generate 확인

목표:
- Qwen3-0.6B의 모든 nn.Linear를 FakeQuantLinear로 교체
- 교체 후 model.generate()가 깨지지 않음을 확인
- scale=None 상태이므로 FP output과 동일해야 함

## 1. 모델 로드

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
model.eval()
print("Model loaded.")

## 2. FP Generate (교체 전 기준값)

In [ ]:
prompt = "Quantization is a technique that"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    fp_output_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)

fp_output = tokenizer.decode(fp_output_ids[0], skip_special_tokens=True)
print("[FP output]")
print(fp_output)

## 3. Linear → FakeQuantLinear 교체

`ignore=["lm_head"]` 적용.

In [ ]:
from mini_compressor.fake_quant_linear import FakeQuantLinear
from mini_compressor.schemes import W8A8

IGNORE = ["lm_head"]

def replace_linear(model, scheme, ignore):
    replaced = 0
    for name, module in list(model.named_modules()):
        if not isinstance(module, nn.Linear):
            continue
        if any(name == ig or name.endswith("." + ig) for ig in ignore):
            continue

        # parent module 찾기
        parts = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)

        setattr(parent, parts[-1], FakeQuantLinear.from_float(module, scheme))
        replaced += 1

    return replaced

n = replace_linear(model, W8A8, IGNORE)
print(f"Replaced {n} Linear layers with FakeQuantLinear.")

## 4. 교체 결과 확인

In [ ]:
fake_quant_layers = [
    name for name, m in model.named_modules()
    if isinstance(m, FakeQuantLinear)
]
remaining_linear = [
    name for name, m in model.named_modules()
    if isinstance(m, nn.Linear) and not isinstance(m, FakeQuantLinear)
]

print(f"FakeQuantLinear: {len(fake_quant_layers)}")
print(f"남은 nn.Linear:  {len(remaining_linear)} → {remaining_linear}")

## 5. 교체 후 Generate 확인

scale=None 상태이므로 FP output과 동일해야 함.

In [ ]:
with torch.no_grad():
    fq_output_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)

fq_output = tokenizer.decode(fq_output_ids[0], skip_special_tokens=True)
print("[FakeQuantLinear output (scale=None)]")
print(fq_output)

print()
match = fp_output == fq_output
print(f"FP output과 동일: {match}")
if not match:
    print("[경고] output이 다름 — forward 로직 확인 필요")

## 6. weight_scale 상태 확인

calibration 전이므로 모든 scale이 None이어야 함.

In [ ]:
sample = next(
    m for _, m in model.named_modules()
    if isinstance(m, FakeQuantLinear)
)

print(f"weight_scale:      {sample.weight_scale}")
print(f"weight_zero_point: {sample.weight_zero_point}")
print(f"input_scale:       {sample.input_scale}")
print(f"scheme:            {sample.scheme.name}")